# Corporate Valuation (DCF) Tear Sheet

Project unlevered free cash flow, run a Gordon-growth DCF, and render a
`dcf_tearsheet` — the EV→equity bridge, the UFCF projection, a WACC /
terminal-growth sensitivity tornado, and a forecast summary.

In [ ]:
import datetime as dt
import json

from finstack_quant import reporting
from finstack_quant.statements import Evaluator, ModelBuilder
from finstack_quant.statements_analytics import evaluate_dcf

qs = ["2025Q1", "2025Q2", "2025Q3", "2025Q4", "2026Q1", "2026Q2", "2026Q3", "2026Q4"]
b = ModelBuilder("acme-dcf")
b.periods("2025Q1..2026Q4", "2025Q2")
b.value("revenue", list(zip(qs, [100, 105, 110, 116, 122, 128, 134, 140])))
b.compute("ebitda", "revenue * 0.27")
b.compute("ufcf", "ebitda * 0.6")
b.compute("net_income", "ebitda * 0.5")
spec = b.build()
result = Evaluator().evaluate(spec)

# evaluate_dcf requires the model's currency in metadata.
raw = json.loads(spec.to_json())
raw.setdefault("meta", {})["currency"] = "USD"
model_json = json.dumps(raw)

TV = json.dumps({"type": "gordon_growth", "growth_rate": 0.025})
NET_DEBT = 51.0
WACC = 0.10

dcf = evaluate_dcf(model_json, WACC, TV, "ufcf", net_debt_override=NET_DEBT)
print("Enterprise value:", round(dcf["enterprise_value"], 1), " Equity value:", round(dcf["equity_value"], 1))


## WACC & terminal-growth sensitivity

Re-run the DCF at shifted WACC and terminal-growth assumptions and bridge the
change in equity value into a tornado (higher WACC ⇒ lower equity).

In [ ]:
def equity_at(wacc, growth):
    tv = json.dumps({"type": "gordon_growth", "growth_rate": growth})
    return evaluate_dcf(model_json, wacc, tv, "ufcf", net_debt_override=NET_DEBT)["equity_value"]

base_eq = dcf["equity_value"]
sensitivity = [
    {
        "parameter_id": "WACC ±1%",
        "downside": equity_at(WACC + 0.01, 0.025) - base_eq,
        "upside": equity_at(WACC - 0.01, 0.025) - base_eq,
    },
    {
        "parameter_id": "Terminal growth ±0.5%",
        "downside": equity_at(WACC, 0.020) - base_eq,
        "upside": equity_at(WACC, 0.030) - base_eq,
    },
]
for s in sensitivity:
    print(f"{s['parameter_id']:26s} down={s['downside']:+.1f}  up={s['upside']:+.1f}")


## Valuation tear sheet

In [ ]:
reporting.dcf_tearsheet(
    dcf,
    results=result,
    sensitivity=sensitivity,
    title="Acme Corp — DCF",
    generated=dt.date(2026, 6, 22),
)


## Saving a standalone HTML file

```python
ts = reporting.dcf_tearsheet(dcf, results=result, sensitivity=sensitivity, generated=dt.date(2026, 6, 22))
ts.save("dcf_tearsheet.html")
```
